# 11 — Breakfast ProcedureVRL Actual Feature Extraction

This notebook starts the actual ProcedureVRL feature extraction stage for Breakfast.

Inputs already prepared:

```text
raw Breakfast videos:
  /content/drive/MyDrive/mmf_tas_lab_data/breakfast_raw_videos

ProcedureVRL checkpoint:
  /content/drive/MyDrive/mmf_tas_lab_data/procedurevrl/checkpoints/checkpoint_epoch_00025.pyth

MS-TCN Breakfast data:
  /content/drive/MyDrive/mmf_tas_lab_data/zenodo_ms_tcn_data/breakfast
```

What this notebook does:

1. Builds a ProcedureVRL test CSV from matched Breakfast videos.
2. Patches ProcedureVRL for Colab:
   - `.avi` raw video support in the HowTo100M loader;
   - `np.int` compatibility issue in `tools/feat_extract.py`.
3. Runs official `tools/feat_extract.py`.
4. Saves raw ProcedureVRL outputs.
5. Converts outputs into MS-TCN-style `.npy` feature files.
6. Builds a coarse MS-TCN-compatible dataset:
   - `features/*.npy`
   - `groundTruth/*.txt`
   - `splits/*.bundle`
   - `mapping.txt`

Important: this extracts **coarse uniform clip-level ProcedureVRL features**, not dense per-frame features.  
For a first run, use `RUN_MODE = "smoke"`.  
For the full split-1 extraction, use `RUN_MODE = "full_split1"`.

## 1. Mount Drive and imports

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import os
import sys
import json
import shutil
import subprocess
import time
from collections import defaultdict

import numpy as np
import pandas as pd

print("Python:", sys.version)
print("cwd:", Path.cwd())

Mounted at /content/drive
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
cwd: /content


## 2. Configuration

In [2]:
DRIVE_ROOT = Path("/content/drive/MyDrive/mmf_tas_lab_data")

BREAKFAST_ROOT = DRIVE_ROOT / "zenodo_ms_tcn_data" / "breakfast"
FEATURE_DIR_I3D = BREAKFAST_ROOT / "features"
GT_DIR = BREAKFAST_ROOT / "groundTruth"
SPLIT_DIR = BREAKFAST_ROOT / "splits"
MAPPING_PATH = BREAKFAST_ROOT / "mapping.txt"

RAW_VIDEO_ROOT = DRIVE_ROOT / "breakfast_raw_videos"
RAW_VIDEO_BASE = RAW_VIDEO_ROOT / "videos"

PROCEDUREVRL_REPO = Path("/content/ProcedureVRL")
PROCEDUREVRL_GIT = "https://github.com/facebookresearch/ProcedureVRL.git"
PROCEDUREVRL_CKPT_PATH = DRIVE_ROOT / "procedurevrl" / "checkpoints" / "checkpoint_epoch_00025.pyth"

OUT_ROOT = DRIVE_ROOT / "text_assisted_tas" / "breakfast" / "procedurevrl"
RUNS_ROOT = OUT_ROOT / "runs"
MANIFEST_DIR = OUT_ROOT / "manifests"
DIAG_DIR = OUT_ROOT / "diagnostics"

for p in [OUT_ROOT, RUNS_ROOT, MANIFEST_DIR, DIAG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SPLIT_ID = 1

# RUN_MODE = "smoke"
RUN_MODE = "full_split1"

CONFIGS = {
    "smoke": {
        "max_videos": 5,
        "num_ensemble_views": 4,
        "batch_size": 4,
        "num_workers": 0,
    },
    "full_split1": {
        "max_videos": None,   # all matched split-1 videos
        "num_ensemble_views": 16,
        "batch_size": 4,
        "num_workers": 2,
    },
}

cfg_run = CONFIGS[RUN_MODE]

RUN_NAME = f"procedurevrl_breakfast_{RUN_MODE}_split{SPLIT_ID}_views{cfg_run['num_ensemble_views']}"
RUN_ROOT = RUNS_ROOT / RUN_NAME
CSV_DIR = RUN_ROOT / "data_csv"
RAW_OUT_DIR = RUN_ROOT / "raw_outputs"
MSTCN_OUT_ROOT = RUN_ROOT / "mstcn_format"
MSTCN_FEATURE_DIR = MSTCN_OUT_ROOT / "features"
MSTCN_GT_DIR = MSTCN_OUT_ROOT / "groundTruth"
MSTCN_SPLIT_DIR = MSTCN_OUT_ROOT / "splits"

for p in [RUN_ROOT, CSV_DIR, RAW_OUT_DIR, MSTCN_OUT_ROOT, MSTCN_FEATURE_DIR, MSTCN_GT_DIR, MSTCN_SPLIT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

PREDICT_NPY = RAW_OUT_DIR / "procedurevrl_video_preds.npy"
RUN_LOG_PATH = RUN_ROOT / "feat_extract_log.txt"

print("RUN_MODE:", RUN_MODE)
print("RUN_ROOT:", RUN_ROOT)
print("PREDICT_NPY:", PREDICT_NPY)
print("MSTCN_OUT_ROOT:", MSTCN_OUT_ROOT)

assert RAW_VIDEO_ROOT.exists(), RAW_VIDEO_ROOT
assert RAW_VIDEO_BASE.exists(), RAW_VIDEO_BASE
assert PROCEDUREVRL_CKPT_PATH.exists(), PROCEDUREVRL_CKPT_PATH
assert GT_DIR.exists(), GT_DIR
assert SPLIT_DIR.exists(), SPLIT_DIR
assert MAPPING_PATH.exists(), MAPPING_PATH

print("checkpoint size MB:", PROCEDUREVRL_CKPT_PATH.stat().st_size / 1024**2)

RUN_MODE: full_split1
RUN_ROOT: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/runs/procedurevrl_breakfast_full_split1_split1_views16
PREDICT_NPY: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/runs/procedurevrl_breakfast_full_split1_split1_views16/raw_outputs/procedurevrl_video_preds.npy
MSTCN_OUT_ROOT: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/runs/procedurevrl_breakfast_full_split1_split1_views16/mstcn_format
checkpoint size MB: 513.5653371810913


## 3. Verify GPU

In [3]:
import torch

print("cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

if RUN_MODE == "full_split1" and not torch.cuda.is_available():
    raise RuntimeError("Full ProcedureVRL extraction should be run on GPU.")

cuda available: True
device: Tesla T4


## 4. Clone or reuse ProcedureVRL and install dependencies

In [4]:
if PROCEDUREVRL_REPO.exists():
    print("ProcedureVRL already exists:", PROCEDUREVRL_REPO)
else:
    !git clone {PROCEDUREVRL_GIT} {PROCEDUREVRL_REPO}

%cd /content/ProcedureVRL

!pip install -q yacs simplejson fvcore iopath decord av einops timm pandas scikit-learn opencv-python ffmpeg-python pytorchvideo ipdb ftfy regex tqdm
!pip uninstall -y clip
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -e . -q

PROCEDUREVRL_LIB = PROCEDUREVRL_REPO / "lib"

for p in [PROCEDUREVRL_REPO, PROCEDUREVRL_LIB]:
    s = str(p)
    if s not in sys.path:
        sys.path.insert(0, s)

os.environ["PYTHONPATH"] = str(PROCEDUREVRL_REPO) + ":" + str(PROCEDUREVRL_LIB) + ":" + os.environ.get("PYTHONPATH", "")

print("ProcedureVRL repo:", PROCEDUREVRL_REPO)
print("lib exists:", PROCEDUREVRL_LIB.exists())
print("PYTHONPATH:", os.environ["PYTHONPATH"].split(":")[:4])

Cloning into '/content/ProcedureVRL'...
remote: Enumerating objects: 168, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 168 (delta 6), reused 12 (delta 4), pack-reused 143 (from 1)
Receiving objects: 100% (168/168), 38.77 MiB | 21.00 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/ProcedureVRL
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 8.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

## 5. Patch ProcedureVRL for Colab and `.avi` videos

In [5]:
# tools/feat_extract.py uses np.int, which breaks with recent NumPy.
feat_extract_path = PROCEDUREVRL_REPO / "tools" / "feat_extract.py"
text = feat_extract_path.read_text()

patched = text.replace("np.int", "int")
if patched != text:
    feat_extract_path.write_text(patched)
    print("Patched np.int -> int in tools/feat_extract.py")
else:
    print("No np.int patch needed in tools/feat_extract.py")

# lib/datasets/howto100m.py does not search .avi by default.
howto_path = PROCEDUREVRL_REPO / "lib" / "datasets" / "howto100m.py"
text = howto_path.read_text()

old = 'for extension in [".webm", ".mkv", ".mp4", ".m4a"]:'
new = 'for extension in [".avi", ".AVI", ".webm", ".mkv", ".mp4", ".m4a"]:'

if old in text:
    text = text.replace(old, new)
    howto_path.write_text(text)
    print("Patched HowTo100M loader to include .avi/.AVI")
else:
    print("AVI extension patch already applied or pattern not found.")

# Quick import diagnostics.
import importlib.util

for mod in ["lib", "lib.datasets", "lib.models", "lib.utils.parser", "decord", "torch"]:
    spec = importlib.util.find_spec(mod)
    print(mod, "->", spec is not None, getattr(spec, "origin", None) if spec else None)

Patched np.int -> int in tools/feat_extract.py
Patched HowTo100M loader to include .avi/.AVI
lib -> True /content/ProcedureVRL/lib/__init__.py
lib.datasets -> True /content/ProcedureVRL/lib/datasets/__init__.py
lib.models -> True /content/ProcedureVRL/lib/models/__init__.py
lib.utils.parser -> True /content/ProcedureVRL/lib/utils/parser.py
decord -> True /usr/local/lib/python3.12/dist-packages/decord/__init__.py
torch -> True /usr/local/lib/python3.12/dist-packages/torch/__init__.py


## 6. Helper functions: mapping, split, raw-video matching

In [6]:
def read_lines(path):
    return [x.strip() for x in Path(path).read_text().splitlines() if x.strip()]

def load_mapping(mapping_path):
    id_to_label = {}
    label_to_id = {}
    for line in read_lines(mapping_path):
        parts = line.split()
        idx = int(parts[0])
        label = parts[1]
        id_to_label[idx] = label
        label_to_id[label] = idx
    return id_to_label, label_to_id

def load_split_ids(path):
    return [Path(x).stem for x in read_lines(path)]

def parse_mstcn_breakfast_id(video_id):
    # Example: P03_cam01_P03_cereals -> P03, cam01, P03_cereals
    parts = video_id.split("_")
    if len(parts) < 4:
        return None, None, None
    return parts[0], parts[1], "_".join(parts[2:])

id_to_label, label_to_id = load_mapping(MAPPING_PATH)
train_ids = load_split_ids(SPLIT_DIR / f"train.split{SPLIT_ID}.bundle")
test_ids = load_split_ids(SPLIT_DIR / f"test.split{SPLIT_ID}.bundle")

raw_video_files = []
for pat in ["*.avi", "*.AVI", "*.mp4", "*.MP4"]:
    raw_video_files.extend(RAW_VIDEO_ROOT.rglob(pat))
raw_video_files = sorted(set(raw_video_files))

raw_by_stem = defaultdict(list)
raw_by_lower_path = {}

for p in raw_video_files:
    raw_by_stem[p.stem].append(p)
    raw_by_stem[p.stem.lower()].append(p)
    raw_by_lower_path[str(p).lower()] = p

def find_video_path(video_id):
    person, view, raw_stem = parse_mstcn_breakfast_id(video_id)
    candidates = []

    if person and view and raw_stem:
        # Direct HF layout.
        exact_candidates = [
            RAW_VIDEO_ROOT / "videos" / person / view / f"{raw_stem}.avi",
            RAW_VIDEO_ROOT / "Videos" / person / view / f"{raw_stem}.avi",
            RAW_VIDEO_ROOT / person / view / f"{raw_stem}.avi",
        ]

        for exact in exact_candidates:
            if exact.exists():
                candidates.append(exact)
            lower = str(exact).lower()
            if lower in raw_by_lower_path:
                candidates.append(raw_by_lower_path[lower])

        # Stereo special case:
        # MS-TCN: P36_stereo01_P36_coffee
        # raw HF: videos/P36/stereo/P36_coffee_ch0.avi
        if view.startswith("stereo"):
            import re
            m = re.search(r"stereo(\d+)", view)
            if m:
                channel_idx = int(m.group(1)) - 1
                stereo_candidates = [
                    RAW_VIDEO_ROOT / "videos" / person / "stereo" / f"{raw_stem}_ch{channel_idx}.avi",
                    RAW_VIDEO_ROOT / "Videos" / person / "stereo" / f"{raw_stem}_ch{channel_idx}.avi",
                    RAW_VIDEO_ROOT / person / "stereo" / f"{raw_stem}_ch{channel_idx}.avi",
                ]
                for c in stereo_candidates:
                    if c.exists():
                        candidates.append(c)
                    lower = str(c).lower()
                    if lower in raw_by_lower_path:
                        candidates.append(raw_by_lower_path[lower])

        for base in [
            RAW_VIDEO_ROOT / "videos" / person,
            RAW_VIDEO_ROOT / "Videos" / person,
            RAW_VIDEO_ROOT / person,
        ]:
            if base.exists():
                candidates.extend(base.rglob(f"{raw_stem}.avi"))
                candidates.extend(base.rglob(f"{raw_stem}.AVI"))

    if not candidates and raw_stem:
        candidates.extend(raw_by_stem.get(raw_stem, []))
        candidates.extend(raw_by_stem.get(raw_stem.lower(), []))

    seen = set()
    out = []
    for c in candidates:
        if c not in seen:
            out.append(c)
            seen.add(c)
    return out

print("classes:", len(id_to_label))
print("train_ids:", len(train_ids))
print("test_ids:", len(test_ids))
print("raw videos:", len(raw_video_files))

classes: 48
train_ids: 1460
test_ids: 252
raw videos: 1989


## 7. Build matched video manifest

In [7]:
rows = []

for split_name, ids in [("train", train_ids), ("test", test_ids)]:
    for video_id in ids:
        person, view, raw_stem = parse_mstcn_breakfast_id(video_id)
        matches = find_video_path(video_id)

        rows.append({
            "split": split_name,
            "video_id": video_id,
            "person": person,
            "view": view,
            "raw_stem": raw_stem,
            "matched": len(matches) > 0,
            "num_matches": len(matches),
            "video_path": str(matches[0]) if matches else "",
            "gt_path": str(GT_DIR / f"{video_id}.txt"),
            "i3d_feature_path": str(FEATURE_DIR_I3D / f"{video_id}.npy"),
        })

df_manifest_all = pd.DataFrame(rows)
df_manifest = df_manifest_all[df_manifest_all["matched"]].copy()

if cfg_run["max_videos"] is not None:
    # Keep train/test mix, but restrict for smoke.
    df_manifest = df_manifest.head(cfg_run["max_videos"]).copy()

display(df_manifest_all.groupby(["split", "matched"]).size().reset_index(name="count"))
print("Matched in full split:", int(df_manifest_all["matched"].sum()), "/", len(df_manifest_all))
print("Used in this run:", len(df_manifest))

manifest_all_path = MANIFEST_DIR / f"breakfast_split{SPLIT_ID}_raw_video_manifest_all.csv"
manifest_used_path = RUN_ROOT / "used_video_manifest.csv"

df_manifest_all.to_csv(manifest_all_path, index=False)
df_manifest.to_csv(manifest_used_path, index=False)

print("Saved full manifest:", manifest_all_path)
print("Saved used manifest:", manifest_used_path)

if len(df_manifest_all[~df_manifest_all["matched"]]) > 0:
    print("Missing raw-video mappings:")
    display(df_manifest_all[~df_manifest_all["matched"]].head(20))

assert len(df_manifest) > 0, "No matched videos for extraction."

,split,matched,count
0,test,True,252
1,train,True,1460


Matched in full split: 1712 / 1712
Used in this run: 1712
Saved full manifest: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/manifests/breakfast_split1_raw_video_manifest_all.csv
Saved used manifest: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/runs/procedurevrl_breakfast_full_split1_split1_views16/used_video_manifest.csv


## 8. Get video durations and build ProcedureVRL CSV

In [8]:
try:
    import decord
    from decord import VideoReader, cpu
except Exception as e:
    raise RuntimeError(f"decord import failed: {e}")

def get_video_duration_seconds(video_path):
    video_path = Path(video_path)
    vr = VideoReader(str(video_path), ctx=cpu(0))
    n_frames = len(vr)
    try:
        fps = float(vr.get_avg_fps())
    except Exception:
        fps = 15.0
    if fps <= 0:
        fps = 15.0
    return n_frames / fps, n_frames, fps

csv_rows = []
duration_rows = []

for _, row in df_manifest.iterrows():
    video_path = Path(row["video_path"])
    duration_sec, n_frames, fps = get_video_duration_seconds(video_path)

    # ProcedureVRL loader receives a path relative to DATA.PATH_PREFIX and without extension.
    # DATA.PATH_PREFIX will be RAW_VIDEO_BASE = .../breakfast_raw_videos/videos
    rel = video_path.relative_to(RAW_VIDEO_BASE)
    rel_no_ext = str(rel.with_suffix(""))

    # test.csv format for len=3:
    # path label duration
    # label is dummy 0; this extraction uses model predictions as features.
    csv_rows.append(f"{rel_no_ext} 0 {duration_sec:.3f}")

    duration_rows.append({
        "video_id": row["video_id"],
        "video_path": str(video_path),
        "rel_no_ext": rel_no_ext,
        "duration_sec": duration_sec,
        "num_raw_frames": n_frames,
        "fps": fps,
        "split": row["split"],
    })

# ProcedureVRL expects test.csv under DATA.PATH_TO_DATA_DIR.
(CSV_DIR / "test.csv").write_text("\n".join(csv_rows) + "\n")

# Also create train/val dummy files for safety, although TEST only is used.
(CSV_DIR / "train.csv").write_text("\n".join(csv_rows[:1]) + "\n")
(CSV_DIR / "val.csv").write_text("\n".join(csv_rows[:1]) + "\n")

df_durations = pd.DataFrame(duration_rows)
df_durations.to_csv(RUN_ROOT / "video_duration_metadata.csv", index=False)

print("Saved ProcedureVRL CSV:", CSV_DIR / "test.csv")
print("rows:", len(csv_rows))
display(df_durations.head())
print((CSV_DIR / "test.csv").read_text().splitlines()[:5])

Saved ProcedureVRL CSV: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/runs/procedurevrl_breakfast_full_split1_split1_views16/data_csv/test.csv
rows: 1712


,video_id,video_path,rel_no_ext,duration_sec,num_raw_frames,fps,split
0,P16_cam01_P16_cereals,/content/drive/MyDrive/mmf_tas_lab_data/breakf...,P16/cam01/P16_cereals,36.600000,549,15.0,train
1,P16_cam01_P16_friedegg,/content/drive/MyDrive/mmf_tas_lab_data/breakf...,P16/cam01/P16_friedegg,462.400000,6936,15.0,train
2,P16_cam01_P16_juice,/content/drive/MyDrive/mmf_tas_lab_data/breakf...,P16/cam01/P16_juice,149.466667,2242,15.0,train
3,P16_cam01_P16_milk,/content/drive/MyDrive/mmf_tas_lab_data/breakf...,P16/cam01/P16_milk,59.200000,888,15.0,train
4,P16_cam01_P16_pancake,/content/drive/MyDrive/mmf_tas_lab_data/breakf...,P16/cam01/P16_pancake,536.266667,8044,15.0,train


['P16/cam01/P16_cereals 0 36.600', 'P16/cam01/P16_friedegg 0 462.400', 'P16/cam01/P16_juice 0 149.467', 'P16/cam01/P16_milk 0 59.200', 'P16/cam01/P16_pancake 0 536.267']


In [9]:
from pathlib import Path

repo = Path("/content/ProcedureVRL")

video_builder_path = repo / "lib" / "models" / "video_model_builder.py"
vit_path = repo / "lib" / "models" / "vit.py"

# Patch 1: import alias in video_model_builder.py
text = video_builder_path.read_text()

old = "from lib.models.vit import vit_base_patch16_224"
new = "from lib.models.vit import vit_base_patch16_224_develop as vit_base_patch16_224"

if old in text and new not in text:
    text = text.replace(old, new)
    video_builder_path.write_text(text)
    print("Patched video_model_builder.py import.")
else:
    print("video_model_builder.py import patch already applied or old pattern not found.")

# Patch 2: extra safety alias inside vit.py
vit_text = vit_path.read_text()

alias_code = """

# Colab / repo compatibility alias.
# Some ProcedureVRL files import vit_base_patch16_224, while this repo version defines vit_base_patch16_224_develop.
try:
    vit_base_patch16_224
except NameError:
    vit_base_patch16_224 = vit_base_patch16_224_develop
"""

if "vit_base_patch16_224 = vit_base_patch16_224_develop" not in vit_text:
    vit_path.write_text(vit_text + alias_code)
    print("Added alias to vit.py.")
else:
    print("vit.py alias already exists.")

# Quick import check
import sys, importlib

sys.path.insert(0, str(repo)) if str(repo) not in sys.path else None

import lib.models.vit as vit
print("has vit_base_patch16_224:", hasattr(vit, "vit_base_patch16_224"))
print("has vit_base_patch16_224_develop:", hasattr(vit, "vit_base_patch16_224_develop"))

from lib.models.vit import vit_base_patch16_224
print("vit import OK:", vit_base_patch16_224)

Patched video_model_builder.py import.
Added alias to vit.py.
has vit_base_patch16_224: True
has vit_base_patch16_224_develop: True
vit import OK: <class 'lib.models.vit.vit_base_patch16_224_develop'>


In [10]:
from pathlib import Path

repo = Path("/content/ProcedureVRL")
(repo / "exps").mkdir(parents=True, exist_ok=True)

print("exps exists:", (repo / "exps").exists())
print("exps path:", repo / "exps")

exps exists: True
exps path: /content/ProcedureVRL/exps


## 9. Run official ProcedureVRL `tools/feat_extract.py`

In [11]:
# Official extraction saves a numpy array to TEST.SAVE_PREDICT_PATH.
# Expected rough shape:
#   [num_videos, TEST.NUM_ENSEMBLE_VIEWS * TEST.NUM_SPATIAL_CROPS, MODEL.NUM_CLASSES]
#
# With the provided checkpoint/config, MODEL.NUM_CLASSES is expected to be 9871.

%cd /content/ProcedureVRL

from pathlib import Path
import os
import sys
import time
import subprocess

# Safety: ProcedureVRL tries to save converted weights here.
Path("/content/ProcedureVRL/exps").mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "tools/feat_extract.py",
    "--cfg", "configs/HowTo100M/procedurevrl_adamw.yaml",

    # Disable training and HowTo100M text/caption pretraining mode.
    "TRAIN.ENABLE", "False",
    "TRAIN.TEXT", "''",
    "DEV.ORDER_PRETRAIN_ENABLED", "False",

    # Test / extraction mode.
    "TEST.ENABLE", "True",
    "TEST.DATASET", "howto100m_develop",
    "TEST.CHECKPOINT_FILE_PATH", str(PROCEDUREVRL_CKPT_PATH),
    "TEST.SAVE_PREDICT_PATH", str(PREDICT_NPY),
    "TEST.NUM_ENSEMBLE_VIEWS", str(cfg_run["num_ensemble_views"]),
    "TEST.NUM_SPATIAL_CROPS", "1",
    "TEST.BATCH_SIZE", str(cfg_run["batch_size"]),

    # Breakfast raw video data prepared as ProcedureVRL-compatible test.csv.
    "DATA.PATH_TO_DATA_DIR", str(CSV_DIR),
    "DATA.PATH_PREFIX", str(RAW_VIDEO_BASE),
    "DATA.PATH_LABEL_SEPARATOR", " ",
    "DATA.NUM_FRAMES", "16",
    "DATA.SAMPLING_RATE", "6",
    "DATA.FD", "3.0",
    "DATA.TEST_CROP_SIZE", "224",
    "DATA.TRAIN_JITTER_SCALES", "[256,320]",
    "DATA.INPUT_CHANNEL_NUM", "[3]",
    "DATA.DECODING_BACKEND", "ffmpeg",

    # ProcedureVRL HowTo100M action/step vocabulary size.
    "MODEL.NUM_CLASSES", "9871",
    "MODEL.HEAD_ACT", "softmax",

    "DATA_LOADER.NUM_WORKERS", str(cfg_run["num_workers"]),
    "DATA_LOADER.PIN_MEMORY", "True",

    "NUM_GPUS", "1",
    "NUM_SHARDS", "1",
    "OUTPUT_DIR", str(RUN_ROOT / "procedurevrl_output"),
    "LOG_MODEL_INFO", "False",
]

# Sanity check: after "--cfg <file>", all remaining arguments must be key-value pairs.
opts = cmd[4:]
print("num override args:", len(opts))
print("even:", len(opts) % 2 == 0)

for i in range(0, len(opts), 2):
    print(repr(opts[i]), "=", repr(opts[i + 1]))

assert len(opts) % 2 == 0, "Override list must be key-value pairs"

print("\nCommand:")
print(" ".join(cmd))

start = time.time()

with open(RUN_LOG_PATH, "w") as log_f:
    process = subprocess.run(
        cmd,
        cwd=str(PROCEDUREVRL_REPO),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env={**os.environ, "PYTHONPATH": os.environ.get("PYTHONPATH", "")},
    )
    log_f.write(process.stdout)

elapsed = (time.time() - start) / 60.0

print("returncode:", process.returncode)
print("elapsed minutes:", elapsed)
print("log saved:", RUN_LOG_PATH)
print("\nLast 120 log lines:")
print("\n".join(process.stdout.splitlines()[-120:]))

if process.returncode != 0:
    raise RuntimeError("ProcedureVRL feat_extract.py failed. Inspect the log above.")

/content/ProcedureVRL
num override args: 56
even: True
'TRAIN.ENABLE' = 'False'
'TRAIN.TEXT' = "''"
'DEV.ORDER_PRETRAIN_ENABLED' = 'False'
'TEST.ENABLE' = 'True'
'TEST.DATASET' = 'howto100m_develop'
'TEST.CHECKPOINT_FILE_PATH' = '/content/drive/MyDrive/mmf_tas_lab_data/procedurevrl/checkpoints/checkpoint_epoch_00025.pyth'
'TEST.SAVE_PREDICT_PATH' = '/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/runs/procedurevrl_breakfast_full_split1_split1_views16/raw_outputs/procedurevrl_video_preds.npy'
'TEST.NUM_ENSEMBLE_VIEWS' = '16'
'TEST.NUM_SPATIAL_CROPS' = '1'
'TEST.BATCH_SIZE' = '4'
'DATA.PATH_TO_DATA_DIR' = '/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/runs/procedurevrl_breakfast_full_split1_split1_views16/data_csv'
'DATA.PATH_PREFIX' = '/content/drive/MyDrive/mmf_tas_lab_data/breakfast_raw_videos/videos'
'DATA.PATH_LABEL_SEPARATOR' = ' '
'DATA.NUM_FRAMES' = '16'
'DATA.SAMPLING_RATE' = '6'
'DATA.FD' = '3.0'
'DATA.TEST_CRO

## 10. Validate raw ProcedureVRL output

In [12]:
assert PREDICT_NPY.exists(), f"Missing output: {PREDICT_NPY}"

video_preds = np.load(PREDICT_NPY)

print("PREDICT_NPY:", PREDICT_NPY)
print("video_preds.shape:", video_preds.shape)
print("dtype:", video_preds.dtype)
print("min/max:", float(np.nanmin(video_preds)), float(np.nanmax(video_preds)))
print("nan count:", int(np.isnan(video_preds).sum()))

expected_videos = len(df_manifest)
expected_views = cfg_run["num_ensemble_views"]

assert video_preds.shape[0] == expected_videos, (video_preds.shape[0], expected_videos)
assert video_preds.shape[1] == expected_views, (video_preds.shape[1], expected_views)

np.save(RAW_OUT_DIR / "procedurevrl_video_preds_validated.npy", video_preds)

summary_raw = {
    "run_mode": RUN_MODE,
    "num_videos": int(video_preds.shape[0]),
    "num_views": int(video_preds.shape[1]),
    "feature_dim": int(video_preds.shape[2]),
    "dtype": str(video_preds.dtype),
    "predict_npy": str(PREDICT_NPY),
}

(RUN_ROOT / "raw_feature_summary.json").write_text(json.dumps(summary_raw, indent=2))
print(json.dumps(summary_raw, indent=2))

PREDICT_NPY: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/runs/procedurevrl_breakfast_full_split1_split1_views16/raw_outputs/procedurevrl_video_preds.npy
video_preds.shape: (1712, 16, 9871)
dtype: float64
min/max: 1.6486968040796413e-11 0.3819521963596344
nan count: 0
{
  "run_mode": "full_split1",
  "num_videos": 1712,
  "num_views": 16,
  "feature_dim": 9871,
  "dtype": "float64",
  "predict_npy": "/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/runs/procedurevrl_breakfast_full_split1_split1_views16/raw_outputs/procedurevrl_video_preds.npy"
}


## 11. Convert raw ProcedureVRL output to per-video MS-TCN-style features

In [13]:
# Convert:
#   video_preds[i] shape [num_views, feature_dim]
# to:
#   features/<video_id>.npy shape [feature_dim, num_views]
#
# This matches the MS-TCN convention [D, T].

feature_rows = []

for i, (_, row) in enumerate(df_manifest.reset_index(drop=True).iterrows()):
    video_id = row["video_id"]

    arr = video_preds[i]              # [T, D]
    arr = arr.astype(np.float32)
    feat = arr.T                      # [D, T]

    out_path = MSTCN_FEATURE_DIR / f"{video_id}.npy"
    np.save(out_path, feat)

    feature_rows.append({
        "video_id": video_id,
        "split": row["split"],
        "feature_path": str(out_path),
        "feature_dim": int(feat.shape[0]),
        "feature_len": int(feat.shape[1]),
        "source_video_path": row["video_path"],
    })

df_feature_manifest = pd.DataFrame(feature_rows)
feature_manifest_path = RUN_ROOT / "mstcn_feature_manifest.csv"
df_feature_manifest.to_csv(feature_manifest_path, index=False)

display(df_feature_manifest.head())
print("Saved features:", len(df_feature_manifest))
print("Feature dim:", df_feature_manifest["feature_dim"].unique())
print("Feature len:", df_feature_manifest["feature_len"].unique())
print("Saved manifest:", feature_manifest_path)

,video_id,split,feature_path,feature_dim,feature_len,source_video_path
0,P16_cam01_P16_cereals,train,/content/drive/MyDrive/mmf_tas_lab_data/text_a...,9871,16,/content/drive/MyDrive/mmf_tas_lab_data/breakf...
1,P16_cam01_P16_friedegg,train,/content/drive/MyDrive/mmf_tas_lab_data/text_a...,9871,16,/content/drive/MyDrive/mmf_tas_lab_data/breakf...
2,P16_cam01_P16_juice,train,/content/drive/MyDrive/mmf_tas_lab_data/text_a...,9871,16,/content/drive/MyDrive/mmf_tas_lab_data/breakf...
3,P16_cam01_P16_milk,train,/content/drive/MyDrive/mmf_tas_lab_data/text_a...,9871,16,/content/drive/MyDrive/mmf_tas_lab_data/breakf...
4,P16_cam01_P16_pancake,train,/content/drive/MyDrive/mmf_tas_lab_data/text_a...,9871,16,/content/drive/MyDrive/mmf_tas_lab_data/breakf...


Saved features: 1712
Feature dim: [9871]
Feature len: [16]
Saved manifest: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/runs/procedurevrl_breakfast_full_split1_split1_views16/mstcn_feature_manifest.csv


## 12. Build coarse MS-TCN labels aligned to ProcedureVRL clip features

The ProcedureVRL features are coarse uniform clips.  
For a first MS-TCN-compatible dataset, we downsample each original Breakfast ground-truth sequence to `num_views` labels.

This gives:

```text
features/<video_id>.npy      shape [D, num_views]
groundTruth/<video_id>.txt   length num_views
```

In [14]:
def read_gt_labels(video_id):
    path = GT_DIR / f"{video_id}.txt"
    labels = [x.strip() for x in path.read_text().splitlines() if x.strip()]
    return labels

def downsample_labels_uniform(labels, target_len):
    if target_len <= 0:
        raise ValueError("target_len must be positive")
    if len(labels) == 0:
        return ["SIL"] * target_len

    # Pick labels at uniform centers.
    positions = (np.arange(target_len) + 0.5) * len(labels) / target_len
    indices = np.clip(positions.astype(int), 0, len(labels) - 1)
    return [labels[int(i)] for i in indices]

gt_rows = []

for _, row in df_feature_manifest.iterrows():
    video_id = row["video_id"]
    target_len = int(row["feature_len"])

    labels = read_gt_labels(video_id)
    sampled = downsample_labels_uniform(labels, target_len)

    out_path = MSTCN_GT_DIR / f"{video_id}.txt"
    out_path.write_text("\n".join(sampled) + "\n")

    gt_rows.append({
        "video_id": video_id,
        "original_gt_len": len(labels),
        "coarse_gt_len": len(sampled),
        "feature_len": target_len,
        "gt_path": str(out_path),
    })

df_gt_manifest = pd.DataFrame(gt_rows)
gt_manifest_path = RUN_ROOT / "coarse_gt_manifest.csv"
df_gt_manifest.to_csv(gt_manifest_path, index=False)

display(df_gt_manifest.head())
print("Saved coarse GT:", len(df_gt_manifest))
print("Length mismatches:", int((df_gt_manifest["coarse_gt_len"] != df_gt_manifest["feature_len"]).sum()))

,video_id,original_gt_len,coarse_gt_len,feature_len,gt_path
0,P16_cam01_P16_cereals,544,16,16,/content/drive/MyDrive/mmf_tas_lab_data/text_a...
1,P16_cam01_P16_friedegg,6932,16,16,/content/drive/MyDrive/mmf_tas_lab_data/text_a...
2,P16_cam01_P16_juice,2238,16,16,/content/drive/MyDrive/mmf_tas_lab_data/text_a...
3,P16_cam01_P16_milk,884,16,16,/content/drive/MyDrive/mmf_tas_lab_data/text_a...
4,P16_cam01_P16_pancake,8040,16,16,/content/drive/MyDrive/mmf_tas_lab_data/text_a...


Saved coarse GT: 1712
Length mismatches: 0


## 13. Write MS-TCN-style splits and mapping

In [15]:
# Copy mapping.
shutil.copy2(MAPPING_PATH, MSTCN_OUT_ROOT / "mapping.txt")

# Write split bundle files using only extracted videos.
for split_name in ["train", "test"]:
    ids = df_feature_manifest[df_feature_manifest["split"] == split_name]["video_id"].tolist()
    out_path = MSTCN_SPLIT_DIR / f"{split_name}.split{SPLIT_ID}.bundle"
    out_path.write_text("\n".join([f"{x}.txt" for x in ids]) + "\n")
    print(split_name, len(ids), "->", out_path)

print("MS-TCN-style dataset root:", MSTCN_OUT_ROOT)
print("features:", len(list(MSTCN_FEATURE_DIR.glob('*.npy'))))
print("groundTruth:", len(list(MSTCN_GT_DIR.glob('*.txt'))))
print("splits:", len(list(MSTCN_SPLIT_DIR.glob('*'))))
print("mapping:", (MSTCN_OUT_ROOT / 'mapping.txt').exists())

train 1460 -> /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/runs/procedurevrl_breakfast_full_split1_split1_views16/mstcn_format/splits/train.split1.bundle
test 252 -> /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/runs/procedurevrl_breakfast_full_split1_split1_views16/mstcn_format/splits/test.split1.bundle
MS-TCN-style dataset root: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/runs/procedurevrl_breakfast_full_split1_split1_views16/mstcn_format
features: 1712
groundTruth: 1712
splits: 2
mapping: True


## 14. Validate feature/label alignment

In [16]:
validation_rows = []

for _, row in df_feature_manifest.iterrows():
    video_id = row["video_id"]
    feat = np.load(MSTCN_FEATURE_DIR / f"{video_id}.npy")
    gt_labels = [x.strip() for x in (MSTCN_GT_DIR / f"{video_id}.txt").read_text().splitlines() if x.strip()]

    validation_rows.append({
        "video_id": video_id,
        "split": row["split"],
        "feature_shape": list(feat.shape),
        "feature_dim": int(feat.shape[0]),
        "feature_len": int(feat.shape[1]),
        "gt_len": len(gt_labels),
        "length_match": int(feat.shape[1]) == len(gt_labels),
    })

df_validation = pd.DataFrame(validation_rows)
validation_path = RUN_ROOT / "mstcn_format_validation.csv"
df_validation.to_csv(validation_path, index=False)

display(df_validation.head())
print("videos:", len(df_validation))
print("length mismatches:", int((~df_validation["length_match"]).sum()))
print("feature_dim unique:", sorted(df_validation["feature_dim"].unique().tolist()))
print("feature_len unique:", sorted(df_validation["feature_len"].unique().tolist()))
print("Saved validation:", validation_path)

assert df_validation["length_match"].all(), "Feature/label length mismatch found."

,video_id,split,feature_shape,feature_dim,feature_len,gt_len,length_match
0,P16_cam01_P16_cereals,train,"[9871, 16]",9871,16,16,True
1,P16_cam01_P16_friedegg,train,"[9871, 16]",9871,16,16,True
2,P16_cam01_P16_juice,train,"[9871, 16]",9871,16,16,True
3,P16_cam01_P16_milk,train,"[9871, 16]",9871,16,16,True
4,P16_cam01_P16_pancake,train,"[9871, 16]",9871,16,16,True


videos: 1712
length mismatches: 0
feature_dim unique: [9871]
feature_len unique: [16]
Saved validation: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/runs/procedurevrl_breakfast_full_split1_split1_views16/mstcn_format_validation.csv


## 15. Optional: ProcedureVRL-style text embeddings for Breakfast action prompts

In [17]:
# Optional but useful for the next aligned text/KD experiment.
# It uses OpenAI CLIP text encoder with prompt templates similar to ProcedureVRL's emb_extract.py.
# Run only if you need aligned text features now.

RUN_TEXT_EMBEDDINGS = False

if RUN_TEXT_EMBEDDINGS:
    !pip install -q git+https://github.com/openai/CLIP.git

    import torch
    import clip

    prompts = [
        'a photo of {}.',
        'a photo of a person {}.',
        'a photo of a person doing {}.',
        'a video of {}.',
        'a video of a person {}.',
        'a video of a person doing {}.',
        'a demonstration of {}.',
        'a demonstration of a person {}.',
        'a demonstration of a person doing {}.',
    ]

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model, preprocess = clip.load("ViT-B/16", device=device)
    model.eval()

    action_prompts = df_mapping["prompt"].astype(str).tolist()
    all_features = []

    with torch.no_grad():
        for action in action_prompts:
            sents = [p.format(action) for p in prompts]
            tokens = clip.tokenize(sents, truncate=True).to(device)
            feats = model.encode_text(tokens).float()
            feats = feats / feats.norm(dim=-1, keepdim=True)
            mean_feat = feats.mean(dim=0)
            mean_feat = mean_feat / mean_feat.norm()
            all_features.append(mean_feat.cpu())

    text_features = torch.stack(all_features, dim=0).numpy().astype(np.float32)

    text_out_dir = OUT_ROOT / "text_embeddings"
    text_out_dir.mkdir(parents=True, exist_ok=True)

    np.save(text_out_dir / "breakfast_procedurevrl_clip_vitb16_text_embeddings.npy", text_features)
    df_mapping.to_csv(text_out_dir / "breakfast_procedurevrl_clip_vitb16_text_metadata.csv", index=False)

    print("text_features:", text_features.shape)
    print("saved to:", text_out_dir)
else:
    print("RUN_TEXT_EMBEDDINGS=False, skipping.")

RUN_TEXT_EMBEDDINGS=False, skipping.


## 16. Final summary

In [18]:
final_summary = {
    "status": "completed" if PREDICT_NPY.exists() else "missing_predictions",
    "run_mode": RUN_MODE,
    "run_name": RUN_NAME,
    "split_id": SPLIT_ID,
    "num_videos_extracted": int(len(df_feature_manifest)),
    "num_train_videos": int((df_feature_manifest["split"] == "train").sum()),
    "num_test_videos": int((df_feature_manifest["split"] == "test").sum()),
    "raw_procedurevrl_output": str(PREDICT_NPY),
    "raw_output_shape": list(video_preds.shape),
    "feature_dim": int(video_preds.shape[2]),
    "feature_len_per_video": int(video_preds.shape[1]),
    "mstcn_dataset_root": str(MSTCN_OUT_ROOT),
    "feature_manifest": str(feature_manifest_path),
    "gt_manifest": str(gt_manifest_path),
    "validation_csv": str(validation_path),
    "run_log": str(RUN_LOG_PATH),
    "note": "ProcedureVRL outputs are coarse clip-level features. Ground-truth labels were downsampled to match feature length.",
}

summary_path = RUN_ROOT / "procedurevrl_extraction_summary.json"
summary_path.write_text(json.dumps(final_summary, indent=2))

print(json.dumps(final_summary, indent=2))
print("\nNext step:")
print("Use MSTCN_OUT_ROOT as the feature dataset root for an MS-TCN baseline or text/KD experiment.")

{
  "status": "completed",
  "run_mode": "full_split1",
  "run_name": "procedurevrl_breakfast_full_split1_split1_views16",
  "split_id": 1,
  "num_videos_extracted": 1712,
  "num_train_videos": 1460,
  "num_test_videos": 252,
  "raw_procedurevrl_output": "/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/runs/procedurevrl_breakfast_full_split1_split1_views16/raw_outputs/procedurevrl_video_preds.npy",
  "raw_output_shape": [
    1712,
    16,
    9871
  ],
  "feature_dim": 9871,
  "feature_len_per_video": 16,
  "mstcn_dataset_root": "/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/runs/procedurevrl_breakfast_full_split1_split1_views16/mstcn_format",
  "feature_manifest": "/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/runs/procedurevrl_breakfast_full_split1_split1_views16/mstcn_feature_manifest.csv",
  "gt_manifest": "/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/proce